In [1]:
#importing necessary libraries
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

In [2]:
# loading the dataset
train = pd.read_csv('train.csv')

In [3]:
# Features and target
# F(X) = Y
# trainX = features
# trainY = target
trainX = train.iloc[:, :-1]
trainY = train.iloc[:, -1]

In [4]:
# Identifying categorical columns
# Categorical columns are those that contain non-numeric data, such as strings or categories. 
# These columns often require special handling, such as one-hot encoding, before they can be used in machine learning models.
# object data type in pandas is used to represent string or categorical data. Therefore, selecting columns with the 'object' data type will give us the categorical columns in the dataset.
categorical_cols = trainX.select_dtypes(include=['object', 'string']).columns
print("Categorical Columns: ", len(categorical_cols))

Categorical Columns:  43


In [5]:
# Handling missing values in categorical columns by filling them with the mode (most frequent value) of each column.
trainX[categorical_cols] = trainX[categorical_cols].fillna(trainX[categorical_cols].mode().iloc[0])

In [6]:
# One-hot encoding of categorical columns
ohe =OneHotEncoder(handle_unknown='ignore',sparse_output=False)

encoded_features = ohe.fit_transform(trainX[categorical_cols])

encoded_df = pd.DataFrame(encoded_features, columns=ohe.get_feature_names_out(categorical_cols))


In [7]:
# Dropping original categorical columns and concatenating the one-hot encoded columns to the original dataframe

trainX = trainX.drop(columns=categorical_cols)

trainX = pd.concat([trainX.reset_index(drop=True),encoded_df.reset_index(drop=True)],axis=1)

print("New feature shape: ", trainX.shape)

New feature shape:  (1460, 288)


In [8]:
# Fill numeric missing values
trainX = trainX.fillna(trainX.median())

In [9]:
# Normalize features
feature_scaler = MinMaxScaler()
norm_trainX = feature_scaler.fit_transform(trainX)

In [10]:
# Normalize target
target_scaler = MinMaxScaler()
norm_trainY = target_scaler.fit_transform(trainY.values.reshape(-1,1))

In [11]:
# Building the ANN model

model = Sequential([Input(shape=(norm_trainX.shape[1],1)),
                    Dense(256, activation='relu'),
                    Dense(256, activation='relu'),
                    Dense(128, activation='relu'),
                    Dense(64, activation='relu'),
                    Dense(32, activation='relu'),
                    Dense(1, activation='linear')
                    ])

In [12]:
# Model Summary
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 288, 256)       │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 288, 256)       │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 288, 128)       │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 288, 64)        │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 288, 32)        │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 288, 1)         │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 109,569 (428.00 KB)

 Trainable params: 109,569 (428.00 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
#model.compile(optimizer=keras.optimizers.Adam(), loss='mean_squared_error', metrics=['mean_absolute_error'])

model.compile(optimizer='sgd', loss='mse',metrics=['mae'])

In [ ]:
# Training the model
history = model.fit(norm_trainX, norm_trainY, epochs=50, batch_size=128, validation_split=0.2, verbose=1)

In [16]:
predictions = model.predict(norm_trainX)

print(predictions.shape)

predictions = predictions.reshape(-1, 1)

final_predictions = target_scaler.inverse_transform(predictions)

print(final_predictions[:10])

46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step
(1460, 288, 1)
[[181203.3 ]
 [179777.7 ]
 [179677.34]
 [180909.17]
 [182914.95]
 [181516.17]
 [185847.03]
 [185037.1 ]
 [179692.69]
 [179671.48]]
